# Embedding baseline with human-labeled examples

Each tagged transcript is encoded as a single vector with `sentence-transformers/bert-base-nli-mean-tokens`. Each level is represented by the mean embedding of its labeled examples, and every transcript gets the level of the nearest mean.

**Data**: a csv with one row per video transcript: its tagged text (`Transcript`), the level given by humans (`ability_label_name`), the number of human ratings (`ability_n`) and, optionally, the reference level instructed to the LLM (`REFERENCE_COLUMN`), levels being Low / Medium / High.

**Protocol** (identical in `human_fgw.ipynb` and `human_embeddings.ipynb`):
* **examples**: the videos rated by more than 4 humans (`EXAMPLES_QUERY = "ability_n > 4"`), whose labels are the most reliable; all of them (`M = None`) or $m$ per level drawn among them;
* **evaluation**: all the other videos;
* the examples are labeled either by the **humans** or by the **reference** level, and every classification is scored against **both** labels: accuracy, macro-F1, quadratic-weighted Cohen's $\kappa$ with a 95% bootstrap CI over the evaluated videos;
* the examples are saved in `splits/` and reloaded, so both methods use exactly the same ones;
* with `M` set, the draw of the examples is repeated `N_REPEATS` times (mean ± SD);
* **unsupervised**: $k$-means with $K = 3$ on the evaluated videos, without any label; each cluster is mapped to a level with the Hungarian algorithm (separately for each label evaluated against). ARI and AMI do not depend on that mapping. Repeated with `N_REPEATS` initializations.

In [ ]:
import os
import re
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

sys.path.append(os.path.realpath("../lib"))
from evaluation import (LEVELS, evaluate, kappa_with_ci, load_labeled_transcripts, load_or_create_pool_examples,
                        map_clusters, plot_confusion)
from sklearn.cluster import KMeans

## Parameters

In [ ]:
LABELS_FILE = "human_labels.csv"
TEXT_COLUMN = "Transcript"
LABEL_COLUMN = "ability_label_name"
REFERENCE_COLUMN = None         # reference (instructed) level; None: a column whose name contains "ref" or "instruct"
DIMENSION = "ability"           # names the splits and outputs

EXAMPLES_QUERY = "ability_n > 4"  # videos used as examples (pandas query on the columns of the csv);
                                 # all the other videos are evaluated
SPLIT_DIR = "splits"            # shared with the other notebook
OUTPUT_DIR = "results"

M = None                        # examples per level drawn among them (None: all of them)
SEED = 0                        # draw of the examples (with M) and k-means initialization
N_REPEATS = 5                   # draws of the examples (with M) and k-means initializations

EMBEDDING_MODEL = "sentence-transformers/bert-base-nli-mean-tokens"
METRIC = "cosine"               # "cosine" or "euclidean", as in embedding_baseline.ipynb
N_KMEANS_INIT = 10              # k-means restarts (unsupervised)

os.makedirs(OUTPUT_DIR, exist_ok=True)

## Labeled videos

In [ ]:
if REFERENCE_COLUMN is None:
    columns = pd.read_csv(LABELS_FILE, nrows=0).columns
    REFERENCE_COLUMN = next((c for c in columns if c not in (TEXT_COLUMN, LABEL_COLUMN)
                             and re.search("ref|instruct", c, re.IGNORECASE)), None)
print("reference column:", REFERENCE_COLUMN)

df = load_labeled_transcripts(LABELS_FILE, TEXT_COLUMN, LABEL_COLUMN, REFERENCE_COLUMN)
LABELS = {"human": "level", **({"reference": "reference"} if REFERENCE_COLUMN else {})}  # label -> column of df

# example videos: rows of the csv selected by EXAMPLES_QUERY (ids are row<k>)
example_rows = pd.read_csv(LABELS_FILE).query(EXAMPLES_QUERY).index
df["role"] = np.where(df["id"].isin([f"row{k}" for k in example_rows]), "examples", "evaluated")
pool = df[df["role"] == "examples"]
test = df[df["role"] == "evaluated"]
print(f"{len(pool)} example videos ({EXAMPLES_QUERY}), {len(test)} evaluated videos")
pd.concat({role: pd.DataFrame({label: part[column].value_counts() for label, column in LABELS.items()})
           for role, part in df.groupby("role")}, axis=1).reindex(LEVELS).fillna(0).astype(int)

### Agreement between the human and the reference levels

How often humans recognize the instructed level, on the evaluated videos: the context for the scores below.

In [ ]:
if "reference" in LABELS:
    agreement = kappa_with_ci(pd.DataFrame({"true": test["reference"], "pred": test["level"]}))
    print(f"exact agreement: {(test['reference'] == test['level']).mean():.3f}, "
          f"kappa = {agreement['kappa']:.2f} [{agreement['kappa_ci_low']:.2f}, {agreement['kappa_ci_high']:.2f}]")
    display(pd.crosstab(test["reference"], test["level"], rownames=["reference"], colnames=["human"])
            .reindex(index=LEVELS, columns=LEVELS, fill_value=0))

## Embeddings

In [ ]:
model = SentenceTransformer(EMBEDDING_MODEL)
X = model.encode(df["text"].tolist(), batch_size=64, show_progress_bar=True)
if METRIC == "cosine":  # cosine distance = Euclidean distance between normalized vectors
    X = X / np.linalg.norm(X, axis=1, keepdims=True)
vectors = dict(zip(df["id"], X))

## Nearest mean embedding

In [ ]:
def classify(folds, examples):
    # Predicted level of every transcript, from the examples of its own fold.
    predictions = []
    for fold in sorted(folds["fold"].unique()):
        test = folds[folds["fold"] == fold]
        support = examples[examples["fold"] == fold]
        centroids = np.array([np.mean([vectors[i] for i in support.loc[support["level"] == level, "id"]], axis=0)
                              for level in LEVELS])
        X_test = np.array([vectors[i] for i in test["id"]])
        distances = np.linalg.norm(X_test[:, None, :] - centroids[None, :, :], axis=2)
        pred = pd.DataFrame({"id": test["id"], "fold": fold, "true": test["level"],
                             "pred": np.array(LEVELS)[distances.argmin(axis=1)]})
        for k, level in enumerate(LEVELS):
            pred[f"distance_{level}"] = distances[:, k]
        predictions.append(pred)
    return pd.concat(predictions, ignore_index=True)


def cluster(ids, seed):
    # k-means (K = 3) on the embeddings of the transcripts of ids.
    labels = KMeans(n_clusters=len(LEVELS), n_init=N_KMEANS_INIT, random_state=seed).fit_predict(
        np.array([vectors[i] for i in ids]))
    return pd.DataFrame({"id": ids, "cluster": labels})

## Classification

Examples labeled by humans or by the reference level; every classification of the evaluated videos is scored against both labels.

In [ ]:
METHOD = "embeddings"
TAG = f"{DIMENSION}_{re.sub(r'[^A-Za-z0-9]+', '', EXAMPLES_QUERY)}"  # names the saved examples
folds = test[["id", "level"]].assign(fold=0)  # a single "fold": the evaluated videos


def run(seed):
    # Predictions with examples labeled by each label (drawn with this seed when M is set).
    predictions = {}
    for source, column in LABELS.items():
        examples = load_or_create_pool_examples(pool["id"], pool[column], f"{TAG}_{source}_seed{seed}",
                                                SPLIT_DIR, M, seed)
        pred = classify(folds, examples).drop(columns=["true", "fold"])
        predictions[source] = pred.merge(df[["id", *LABELS.values()]], on="id")
    return predictions


def score_all(predictions):
    # Scores of every classification against every label.
    results = []
    for source, pred in predictions.items():
        for target, column in LABELS.items():
            results.append(evaluate({(DIMENSION, f"examples: {source}"): pred.assign(true=pred[column])},
                                    method=METHOD, reference=target))
    return pd.concat(results, ignore_index=True).rename(columns={"setting": "examples", "reference": "evaluated against"})


predictions = run(SEED)
for source, pred in predictions.items():
    pred.to_csv(os.path.join(OUTPUT_DIR, f"human_embedding_predictions_{DIMENSION}_{source}_examples.csv"), index=False)
results = score_all(predictions).drop(columns="fold")
results.to_csv(os.path.join(OUTPUT_DIR, f"human_embedding_results_{DIMENSION}.csv"), index=False)
results["kappa [95% CI]"] = [f"{k:.2f} [{lo:.2f}, {hi:.2f}]" for k, lo, hi in
                             zip(results["kappa"], results["kappa_ci_low"], results["kappa_ci_high"])]
results.pivot(index="examples", columns="evaluated against", values=["accuracy", "macro_f1", "kappa [95% CI]"])

Confusion matrices (rows: label evaluated against; columns: examples used).

In [ ]:
plot_confusion({(f"vs {target}", f"examples: {source}"): pred.assign(true=pred[column])
                for target, column in LABELS.items() for source, pred in predictions.items()},
               "true", os.path.join(OUTPUT_DIR, f"human_embedding_confusion_{DIMENSION}.png"))
plt.show()

## Variability over draws of the examples

Only with `M` set: the examples are drawn `N_REPEATS` times among the example videos (the evaluated videos do not change).

In [ ]:
if M is None:
    repeats = results.assign(seed=SEED)[["seed", "method", "examples", "evaluated against", "accuracy", "macro_f1", "kappa"]]
    print("M = None: all the example videos are used, there is nothing to draw.")
else:
    repeats = []
    for seed in range(SEED, SEED + N_REPEATS):
        repeats.append(score_all(predictions if seed == SEED else run(seed)).query("fold == 'all'").assign(seed=seed))
    repeats = pd.concat(repeats, ignore_index=True)[["seed", "method", "examples", "evaluated against",
                                                     "accuracy", "macro_f1", "kappa"]]
    display(repeats.groupby(["examples", "evaluated against"])[["accuracy", "macro_f1", "kappa"]]
            .agg(["mean", "std"]).round(3))
repeats.to_csv(os.path.join(OUTPUT_DIR, f"human_embedding_repeats_{DIMENSION}.csv"), index=False)

## Unsupervised: $k$-means and Hungarian mapping

On the evaluated videos (the same as above), without any label; the labels only map clusters to levels for scoring.

In [ ]:
UNSUPERVISED = "none (k-means)"


def run_unsupervised(seed):
    return cluster(list(test["id"]), seed).merge(df[["id", *LABELS.values()]], on="id")


def score_unsupervised(pred):
    # Scores against every label, each with its own cluster-to-level mapping.
    results = []
    for target, column in LABELS.items():
        mapped = pred.assign(true=pred[column], pred=map_clusters(pred["cluster"], pred[column]))
        results.append(evaluate({(DIMENSION, UNSUPERVISED): mapped}, method=METHOD, reference=target))
    return pd.concat(results, ignore_index=True).rename(columns={"setting": "examples", "reference": "evaluated against"})


unsupervised = run_unsupervised(SEED)
unsupervised.to_csv(os.path.join(OUTPUT_DIR, f"human_embedding_unsupervised_predictions_{DIMENSION}.csv"), index=False)
unsupervised_results = score_unsupervised(unsupervised).drop(columns="fold")
unsupervised_results.to_csv(os.path.join(OUTPUT_DIR, f"human_embedding_unsupervised_results_{DIMENSION}.csv"), index=False)
unsupervised_results["kappa [95% CI]"] = [f"{k:.2f} [{lo:.2f}, {hi:.2f}]" for k, lo, hi in zip(
    unsupervised_results["kappa"], unsupervised_results["kappa_ci_low"], unsupervised_results["kappa_ci_high"])]
unsupervised_results[["evaluated against", "accuracy", "macro_f1", "kappa [95% CI]", "ARI", "AMI"]].round(3)

In [ ]:
plot_confusion({(f"vs {target}", UNSUPERVISED):
                unsupervised.assign(true=unsupervised[column], pred=map_clusters(unsupervised["cluster"], unsupervised[column]))
                for target, column in LABELS.items()},
               "true", os.path.join(OUTPUT_DIR, f"human_embedding_unsupervised_confusion_{DIMENSION}.png"))
plt.show()

Over `N_REPEATS` initializations of $k$-means:

In [ ]:
unsupervised_repeats = []
for seed in range(SEED, SEED + N_REPEATS):
    seed_results = score_unsupervised(unsupervised if seed == SEED else run_unsupervised(seed))
    unsupervised_repeats.append(seed_results.assign(seed=seed))
unsupervised_repeats = pd.concat(unsupervised_repeats, ignore_index=True)[
    ["seed", "method", "examples", "evaluated against", "accuracy", "macro_f1", "kappa", "ARI", "AMI"]]
unsupervised_repeats.to_csv(os.path.join(OUTPUT_DIR, f"human_embedding_unsupervised_repeats_{DIMENSION}.csv"), index=False)
unsupervised_repeats.groupby("evaluated against")[["accuracy", "macro_f1", "kappa", "ARI", "AMI"]].agg(["mean", "std"]).round(3)